# ***Tenno: Data Privacy Act***

## **Hermes-7B Baseline Evaluation**

This notebook evaluates Nous Hermes under a Retrieval-Augmented Generation (RAG) setup using the same FAISS retrieval pipeline and the same frozen baseline prompts. For each prompt, the system retrieves top-k relevant chunks, builds a grounded context, and generates an answer. Outputs are saved to Google Drive as JSON for later comparison across models.

## **Install Required Packages**

---


In [1]:
!pip install -q transformers accelerate bitsandbytes faiss-cpu sentence-transformers tqdm

Installs the libraries required to load Hermes, load the saved FAISS index, retrieve relevant chunks, and run inference in Colab.

## **Mount Drive**

---


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Mounts Google Drive so the notebook can read the frozen prompts and FAISS retrieval artifacts and save model outputs back to Drive.

## **Persistent Hugging Face CachePersistent Hugging Face Cache**

---


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project"
HF_CACHE_DIR = f"{PROJECT_DIR}/Models/hf_cache"  # persistent cache in Drive
os.makedirs(HF_CACHE_DIR, exist_ok=True)

# Force HF/Transformers to use Drive cache
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

print("✅ Using persistent HF cache:", HF_CACHE_DIR)

✅ Using persistent HF cache: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/hf_cache


Ensures model downloads are cached in Google Drive so you don’t re-download weights every session.

## **Define Paths + Load Prompts and FAISS Artifacts**

---


In [4]:
import os, json
import faiss

PROJECT_DIR = "/content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project"

PROMPTS_FILE = f"{PROJECT_DIR}/Prompts/baseline_prompts_v1.json"
FAISS_INDEX_PATH = f"{PROJECT_DIR}/Chunks/faiss_index.bin"
FAISS_META_PATH = f"{PROJECT_DIR}/Chunks/faiss_metadata.json"

RESULTS_DIR = f"{PROJECT_DIR}/Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

OUTPUT_PATH = f"{RESULTS_DIR}/baseline_outputs_hermes.json"

# Load prompts
with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompts = json.load(f)

# Load FAISS index
index = faiss.read_index(FAISS_INDEX_PATH)

# Load metadata (must align with index vector order)
with open(FAISS_META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("✅ Prompts loaded:", len(prompts))
print("✅ FAISS vectors:", index.ntotal)
print("✅ Metadata rows:", len(meta))
print("✅ Output path:", OUTPUT_PATH)

✅ Prompts loaded: 10
✅ FAISS vectors: 198
✅ Metadata rows: 198
✅ Output path: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Results/baseline_outputs_hermes.json


Loads the frozen prompt list, the saved FAISS vector index, and chunk metadata needed for RAG retrieval.

## **Load SentenceTransformer for Query Embedding**

---


In [5]:
from sentence_transformers import SentenceTransformer

embed_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

print(f"✅ Query embedder loaded: {embed_model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Query embedder loaded: all-MiniLM-L6-v2


Embeds questions into the same vector space used to build the FAISS inde

## **Retrieval Helper (Top-k Chunks)**

---


In [6]:
import numpy as np

TOP_K = 3

def retrieve_top_k_chunks(query: str, top_k: int = TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = q_emb.astype(np.float32)

    distances, indices = index.search(q_emb, top_k)
    idxs = indices[0].tolist()

    retrieved = []
    for rank, i in enumerate(idxs):
        row = meta[i]
        retrieved.append({
            "faiss_id": i,
            "text": row.get("text", ""),
            "source": row.get("source", ""),
            "page_number": row.get("page_number", None),
            "distance": float(distances[0][rank])
        })
    return retrieved

Retrieves the top-k most relevant legal chunks for each question from the FAISS index.

## **Load Hermes-7B Model (4-bit)**

---


In [7]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading Hermes tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Memory cleanup BEFORE loading model
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Loading Hermes model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    device_map={"": 0},
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    force_download=True
)

model.eval()
print("✅ Hermes loaded in 4-bit.")

Loading Hermes tokenizer...
Loading Hermes model (4-bit)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

✅ Hermes loaded in 4-bit.


Loads Hermes using 4-bit quantization for Colab stability (same approach as Falcon/Qwen).

## **Build Model Input (Context + Question)**

---


In [12]:
def build_rag_input(retrieved_chunks, question: str) -> str:
    context_blocks = []
    for c in retrieved_chunks:
        src = c["source"]
        pg = c["page_number"]
        txt = c["text"]
        context_blocks.append(f"[Source: {src} | Page: {pg}]\n{txt}")

    context = "\n\n---\n\n".join(context_blocks)

    return (
        "You are a legal assistant. Answer the question using ONLY the retrieved context.\n\n"
        f"Retrieved Context (Top-{TOP_K}):\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

Formats retrieved chunks into a grounded context block and appends the question.

## **Deterministic Generation Helper**

---


In [9]:
@torch.inference_mode()
def generate_answer(input_text: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    out_ids = model.generate(
        **inputs,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return decoded[len(input_text):].strip()

## **Run All Prompts and Save Outputs**

---


In [10]:
from tqdm import tqdm
import json

results = []

for p in tqdm(prompts, desc="Running Hermes RAG prompts"):
    q = p["question"]

    retrieved = retrieve_top_k_chunks(q, top_k=TOP_K)
    input_text = build_rag_input(retrieved, q)
    answer = generate_answer(input_text, max_new_tokens=256)

    results.append({
        "model_id": MODEL_ID,
        "prompt_id": p.get("prompt_id"),
        "title": p.get("title"),
        "method": p.get("method"),
        "condition": p.get("condition"),
        "dataset_file": p.get("dataset_file"),
        "section_index": p.get("section_index"),
        "question": q,
        "retrieved_chunks": retrieved,
        "answer": answer,
        "expected_output": p.get("expected_output")
    })

# Save
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ Saved outputs to:", OUTPUT_PATH)

Running Hermes RAG prompts: 100%|██████████| 10/10 [04:54<00:00, 29.45s/it]


✅ Saved outputs to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Results/baseline_outputs_hermes.json


Runs Hermes across all 10 prompts using the same retrieval pipeline and saves outputs for later comparison.

## **Display Outputs**

---


In [11]:
import json

RESULTS_PATH = f"{PROJECT_DIR}/Results/baseline_outputs_hermes.json"

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"\n✅ Loaded {len(results)} Hermes outputs\n")

for item in results:
    print("=" * 80)
    print(f"[Prompt {item['prompt_id']}] ({item['method']})")
    print("-" * 80)
    print("Q:", item["question"])
    print("\nA:", item["answer"][:400])
    print("\n")


✅ Loaded 10 Hermes outputs

[Prompt 1] (Zero-Shot)
--------------------------------------------------------------------------------
Q: List all the conditions under which the processing of personal information is considered lawful according to this section.

A: n form at ion is in ad mi ss ib le . ch ap ter iv ri gh ts of the data sub ject sec . 1 6 . ri gh ts of the data sub ject . the data sub ject is ent it led to : ( a ) be in form ed wh et her per son al in form at ion per ta in ing to him or her sh all be , are be ing or ha ve be en pr oc ess ed ; ( b ) be f urn ish ed the in form at ion ind ic at ed he re und er be fo re the ent ry of his or her p


[Prompt 2] (Zero-Shot)
--------------------------------------------------------------------------------
Q: What are the penalties for unauthorized disclosure of personal information and sensitive personal information under this section?

A: em pl oy ees co mp ly with su ch re qu ir em ents . ch ap ter vi ii pe na lt ies sec . 2 5 . 

Prints a compact preview of each Hermes-generated answer to verify quality before moving on.